In [1]:
from gemini_ren import *

/home/cz/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import numpy as np
import time
from tqdm import tqdm
import random

In [3]:
os.environ['nfewshot'] = '5'
# for tracking, put here values used previously: 
nfewshot_value = int(os.environ.get('nfewshot', 0))

In [4]:
root = '/home/cz/'

In [5]:
pathx = os.path.join(root, 'mds3/REN/datasets_zeroshot/prepared')

In [6]:
path2save = os.path.join(root, f'mds3/REN/few-shot/OUTPUTS/{nfewshot_value}shot/Gemini/')
os.makedirs(path2save,exist_ok=True)

In [7]:
texts = np.load(pathx+'/texts_test_v1.npy',allow_pickle=True)

In [8]:
targets = np.load(pathx+'/labels_test_v1.npy',allow_pickle=True).item()

In [13]:
for key in targets.keys():
    print(targets[key].keys())

dict_keys(['ORG', 'PESSOA'])
dict_keys(['ORG', 'LOC'])
dict_keys(['LOC', 'PESSOA', 'ORG'])
dict_keys(['PESSOA', 'LOC', 'ORG'])
dict_keys(['LOC', 'PESSOA', 'ORG'])
dict_keys(['PESSOA', 'LOC', 'ORG'])
dict_keys(['ORG', 'PESSOA', 'LOC'])
dict_keys(['PESSOA', 'ORG'])
dict_keys(['LOC', 'PESSOA', 'ORG'])
dict_keys(['ORG', 'PESSOA', 'LOC'])
dict_keys(['ORG', 'PESSOA', 'LOC'])
dict_keys(['LOC', 'PESSOA', 'ORG'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['ORG'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['ORG', 'PESSOA', 'LOC'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['ORG'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['PESSOA', 'LOC', 'ORG'])
dict_keys(['ORG', 'LOC'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['ORG', 'PESSOA', 'LOC'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['ORG'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_keys(['PESSOA', 'LOC', 'ORG'])
dict_keys(['PESSOA', 'ORG', 'LOC'])
dict_

In [9]:
examples = np.load(os.path.join(pathx,'fewshot', f'amostras_selecionadas_{nfewshot_value}.npy'),allow_pickle=True)

In [10]:
file_name='teste.txt'
with open(file_name, 'w', encoding='utf-8') as file:
    for line in texts:
        file.write(line + '\n')  # Escreve cada linha seguida de uma nova linha


In [11]:
def extract_entities_fewshot(text: str, examples:list) -> list[Entity]:
    # Usando um modelo de NER
    model = genai.GenerativeModel("gemini-1.5-pro-latest")
    
    # Geração do conteúdo
    resultado = model.generate_content(
        f"Identifique as entidades nomeadas no texto {text}, baseando-se nos exemplos:{examples}\n. RETORNE UMA LISTA DE TUPLAS com o formato (ENTIDADE, LABEL).",
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json"
        )
    )
    #print(resultado)
    # Assuming 'resultado' is your GenerateContentResponse object
    try:
        # Accessing the text directly
        output = resultado.candidates[0].content.parts[0].text
        #print(output)
    except Exception as e:
        print("An error occurred:", e)
    return output

In [12]:
checkpoint_file = os.path.join(path2save, 'checkpoint.txt')

# Carregar o índice de onde continuar
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        start_index = int(f.read().strip())+1
else:
    start_index = 0

# Loop através dos textos, começando do índice salvo
for i in tqdm(range(start_index, len(texts))):
    text = texts[i]
    entities = extract_entities_fewshot(text=text,examples=examples )
    print(entities)

     # Loop para tentar converter até ter sucesso
    try:
        #predicted_entities = converting_fewshot(entities)

        # Salvar as entidades preditas
        with open(os.path.join(path2save, f'predicted_entities-{i}.txt'), 'w') as f:
            f.write(entities)


        # Salvar o índice atual no checkpoint
        with open(checkpoint_file, 'w') as f:
            f.write(str(i))

    except Exception as e:
        print(f"Erro ao converter entidades: {e}. Tentando novamente...")
        time.sleep(5)  # Espera um pouco antes de tentar novamente
        continue
    time.sleep(45)  # Espera antes de passar para o próximo texto



  0%|          | 0/13 [00:00<?, ?it/s]

{"entities": [["desembargador Ney Bello", "PESSOA"], ["Tribunal Regional Federal da 1ª Região", "ORG"], ["TRF-1", "ORG"], ["Ministério Público Federal", "ORG"], ["MPF", "ORG"], ["Paulo Guedes", "PESSOA"], ["Metrópoles", "ORG"], ["Operação Greenfield", "LEI"], ["Comissão de Valores Mobiliários", "ORG"], ["CVM", "ORG"], ["MPF", "ORG"], ["3ª Turma do tribunal", "ORG"]]}



  8%|▊         | 1/13 [00:49<09:53, 49.44s/it]

[["Esther Morales", "PESSOA"], ["Evo Morales", "PESSOA"], ["Ponciano Willcarani", "PESSOA"], ["Esther", "PESSOA"], ["Esther Morales", "PESSOA"], ["Evo Morales", "PESSOA"], ["Movimento ao Socialismo", "ORG"], ["MAS", "ORG"], ["Evo Morales", "PESSOA"], ["COVID-19", "DOENÇA"], ["La Patria de Oruro", "ORG"], ["Bolívia", "LOC"], ["Bolívia", "LOC"], ["Argentina", "LOC"], ["Oruro", "LOC"], ["Bolívia", "LOC"]]



 15%|█▌        | 2/13 [01:38<08:58, 48.99s/it]

...

 85%|████████▍ | 11/13 [09:34<01:41, 50.96s/it]

[["Câmara Municipal dos Vereadores de Bom Princípio", "ORG"], ["Bom Princípio", "LOC"], ["Norte do Piauí", "LOC"], ["segunda-feira", "DATA"], ["10/08", "DATA"], ["Genycleson de Sousa Galeno", "PESSOA"], ["PT", "ORG"], ["Portal do Rurik", "ORG"], ["Banco do Brasil", "ORG"], ["Fernando do Nascimento", "PESSOA"], ["Delegacia da Policia Civil", "ORG"], ["Parnaíba", "LOC"], ["Jacinto Costa Moraes", "PESSOA"], ["Pedro Neto", "PESSOA"], ["Iracema", "PESSOA"], ["Padinha", "PESSOA"], ["Portal do Rurik", "ORG"], ["30 dias", "TEMPO"]]



 92%|█████████▏| 12/13 [10:23<00:50, 50.48s/it]

[{"ENTIDADE": "Francisco Erismar Jorge da Costa", "LABEL": "PESSOA"}, {"ENTIDADE": "Erismar Jorge", "LABEL": "PESSOA"}, {"ENTIDADE": "Prefeitura de Aroazes", "LABEL": "ORG"}, {"ENTIDADE": "Tomé Portela", "LABEL": "PESSOA"}, {"ENTIDADE": "Covid-19", "LABEL": "DOENÇA"}, {"ENTIDADE": "Portal V1", "LABEL": "ORG"}, {"ENTIDADE": "Prefeito Tomé", "LABEL": "PESSOA"}, {"ENTIDADE": "Erismar", "LABEL": "PESSOA"}, {"ENTIDADE": "Manoel Filho", "LABEL": "PESSOA"}, {"ENTIDADE": "Tomé Portela", "LABEL": "PESSOA"}, {"ENTIDADE": "Centro Covid do município", "LABEL": "LOCAL"}, {"ENTIDADE": "Aroazes", "LABEL": "LOCAL"}, {"ENTIDADE": "Teresina", "LABEL": "LOCAL"}, {"ENTIDADE": "Lei Orgânica", "LABEL": "LEI"}, {"ENTIDADE": "Câmara de Vereadores", "LABEL": "ORG"}, {"ENTIDADE": "Erismar Jorge", "LABEL": "PESSOA"}, {"ENTIDADE": "Lei Orgânica", "LABEL": "LEI"}, {"ENTIDADE": "Abimael Costa", "LABEL": "PESSOA"}, {"ENTIDADE": "Manoel Portela", "LABEL": "PESSOA"}, {"ENTIDADE": "Erismar Jorge", "LABEL": "PESSOA"}, {

100%|██████████| 13/13 [11:17<00:00, 52.09s/it]


In [11]:
root = root = '/home/cz/'

In [12]:
import sys
sys.path.append(root+'/mds3/REN/few-shot')
from help2process_coral import*

In [23]:
text = texts[35]
entities = extract_entities_fewshot(text=text,examples=examples )
print(entities)

[["estados", "LOC"], ["pandemia", "MISC"], ["Covid-19", "MISC"], ["G1", "ORG"], ["país", "LOC"], ["Distrito Federal", "LOC"], ["Lei de Acesso à Informação", "MISC"], ["fim do mês de junho", "MISC"], ["Covid-19", "MISC"], ["UTI", "MISC"], ["Brasil", "LOC"], ["Ministério Público de Contas", "ORG"], ["Tribunais de Contas dos Estados", "ORG"], ["Rio de Janeiro", "LOC"], ["Santa Catarina", "LOC"], ["Pará", "LOC"], ["Helder Barbalho", "PER"], ["MDB", "ORG"], ["Conselho Nacional dos Secretários de Saúde", "ORG"], ["Conass", "ORG"], ["Alberto Beltrame", "PER"], ["Polícia Federal", "ORG"], ["Ministério Público Federal", "ORG"], ["Consórcio Nordeste", "ORG"], ["Polícia Civil da Bahia", "ORG"], ["Ragnarok", "MISC"], ["Rio de Janeiro", "LOC"], ["São Paulo", "LOC"], ["Paraná", "LOC"], ["Roraima", "LOC"], ["Margareth Portela", "PER"], ["Fiocruz", "ORG"], ["Escola Nacional de Saúde Pública", "ORG"], ["Brasil", "LOC"], ["Nordeste", "LOC"], ["Estados Unidos", "LOC"], ["Rio Grande do Sul", "LOC"], ["Ser

In [ ]:
# Após concluir, remover o arquivo de checkpoint
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)

In [ ]:
cla = {"LOC": ["China", "São Paulo", "São Paulo", "EUA", "Rio", "China", "Brasil", "China", "Brasil", "Nova York", "Paraná", "Detran", "Etiópia", "Londres", "Amsterdã", "Etiópia", "Guangzhou", "Brasil", "Rio de Janeiro", "Santa Catarina", "Roraima", "Amazonas"], "ORG": ["Folha", "PSDB", "UTI", "Folha", "Deus", "Folha", "Hichens Harrison & Co.", "Ethernity", "Comen", "Ministério Público", "TCE", "Tribunal de Contas do Estado", "Folha", "Azul", "Ministério Público", "Promotoria do Paraná", "PTB", "Lava Jato", "Procuradoria do Estado", "Folha", "Secretaria de Estado da Saúde", "Hichens", "Hichens", "Hitchens", "Shenzen Comen", "Aeroporto Internacional de Guangzhou", "Hichens Harrison & Co.", "MP-RJ", "The Intercept Brasil", "Veigamed", "Polícia Civil", "UOL ", "Procuradoria-Geral da República"], "PESSOA": ["João Doria", "Basile Pantazis", "Basile Pantazis", "Doria", "Gim Argello", "Rodrigo Garcia", "Doria", "Fabiano Kempfer", "Basile Pantazins", "Basile Pantazis", "Pantazis", "Gabriell Neves", "Helton Zeferino", "Francisco Monteiro", "Wilson Lima"]}

In [ ]:
data_dict = cla

In [ ]:

configured_entities = {(label, entity) for label in data_dict.keys() for entity in data_dict[label]}

In [24]:
new_data = caso18_v2(entities)

In [25]:
new_data

[[('estados', 'LOC')],
 [('pandemia', 'MISC')],
 [('Covid-19', 'MISC')],
 [('G1', 'ORG')],
 [('país', 'LOC')],
 [('Distrito Federal', 'LOC')],
 [('Lei de Acesso à Informação', 'MISC')],
 [('fim do mês de junho', 'MISC')],
 [('Covid-19', 'MISC')],
 [('UTI', 'MISC')],
 [('Brasil', 'LOC')],
 [('Ministério Público de Contas', 'ORG')],
 [('Tribunais de Contas dos Estados', 'ORG')],
 [('Rio de Janeiro', 'LOC')],
 [('Santa Catarina', 'LOC')],
 [('Pará', 'LOC')],
 [('Helder Barbalho', 'PER')],
 [('MDB', 'ORG')],
 [('Conselho Nacional dos Secretários de Saúde', 'ORG')],
 [('Conass', 'ORG')],
 [('Alberto Beltrame', 'PER')],
 [('Polícia Federal', 'ORG')],
 [('Ministério Público Federal', 'ORG')],
 [('Consórcio Nordeste', 'ORG')],
 [('Polícia Civil da Bahia', 'ORG')],
 [('Ragnarok', 'MISC')],
 [('Rio de Janeiro', 'LOC')],
 [('São Paulo', 'LOC')],
 [('Paraná', 'LOC')],
 [('Roraima', 'LOC')],
 [('Margareth Portela', 'PER')],
 [('Fiocruz', 'ORG')],
 [('Escola Nacional de Saúde Pública', 'ORG')],
 [('

In [21]:
new_data = [["Doria", "PESSOA"], ["João Doria", "PESSOA"], ["Doria", "PESSOA"], ["Doria", "PESSOA"], ["Márcio Nakashima", "PESSOA"], ["Adriana Borgo", "PESSOA"], ["Telhada", "PESSOA"], ["São Paulo", "LOC"], ["São Paulo", "LOC"], ["Itapevi", "LOC"], ["Região Metropolitana de São Paulo", "LOC"], ["SP", "LOC"], ["SP", "LOC"], ["Twitter", "ORG"], ["Corregedoria Extraordinária", "ORG"], ["Cadastro Nacional da Pessoa Jurídica", "ORG"], ["CNPJ", "ORG"], ["Tribunal de Contas", "ORG"], ["Secretaria Estadual da Saúde", "ORG"], ["Diário Oficial", "ORG"], ["Saúde", "ORG"], ["Secretaria Estadual da Saúde", "ORG"], ["Diário Oficial do Estado", "ORG"], ["Anvisa", "ORG"], ["Anvisa", "ORG"]]

In [31]:
path2save= '/home/cz/mds3/REN/few-shot/OUTPUTS/3shot/Gemini'

In [38]:
path2save

'/home/cz/mds3/REN/few-shot/OUTPUTS/5shot/Gemini/Tokenized'

In [26]:
np.save(os.path.join(path2save, f'predicted_entities-{35}.npy'), np.concatenate(new_data))

In [40]:
d =np.load(os.path.join(path2save, f'predicted_entities-{0}.npy'),allow_pickle=True)
print(d)

[['LOC' 'Goiás']
 ['ORG' 'Animale']
 ['PESSOA' 'Rafael Bossolani']
 ['ORG' 'Soma']
 ['PESSOA' 'Thiago']
 ['PESSOA' 'Thiago Hering']
 ['ORG' 'Hering']
 ['ORG' 'Farm']
 ['ORG' 'Arezzo']]


In [215]:
path = '/home/cz/mds3/REN/few-shot/OUTPUTS/Zeroshot/Gemini'

In [217]:
for i in range(64):
    try:
        d =np.load(os.path.join(path, f'predicted_entities-{i}.npy'),allow_pickle=True)
        print(i,d)
        new_d = list(d)
    except  Exception as e:
        print(f"Erro ao converter entidades: {e}. Tentando novamente...")
        print(list(d.item()))
        np.save(os.path.join(path, f'predicted_entities-{i}.npy'), list(d.item()))
        

0 {('ORG', 'Animale'), ('ORG', 'Soma'), ('PESSOA', 'Rafael Bossolani'), ('ORG', 'Hering'), ('PESSOA', 'Thiago Hering'), ('ORG', 'Farm'), ('ORG', 'Arezzo')}
Erro ao converter entidades: iteration over a 0-d array. Tentando novamente...
[('ORG', 'Animale'), ('ORG', 'Soma'), ('PESSOA', 'Rafael Bossolani'), ('ORG', 'Hering'), ('PESSOA', 'Thiago Hering'), ('ORG', 'Farm'), ('ORG', 'Arezzo')]
1 {('ORG', 'Associação Brasileira de Proteína Animal'), ('ORG', 'Yara'), ('ORG', 'ABPA'), ('LOC', 'Santa Catarina'), ('LOC', 'Rio Grande do Sul')}
Erro ao converter entidades: iteration over a 0-d array. Tentando novamente...
[('ORG', 'Associação Brasileira de Proteína Animal'), ('ORG', 'Yara'), ('ORG', 'ABPA'), ('LOC', 'Santa Catarina'), ('LOC', 'Rio Grande do Sul')]
2 {('PESSOA', 'Otaviano Canuto'), ('ORG', 'Center for Macroeconomics and Development'), ('ORG', 'Banco Mundial'), ('LOC', 'Chile'), ('ORG', 'Federal Reserve'), ('LOC', 'Washington'), ('ORG', 'BC'), ('PESSOA', 'Joe Biden'), ('ORG', 'Copom'),

In [106]:
list(d.item())

[('PESSOA', 'Luiz Henrique Mandetta'),
 ('ORG', 'CPI da Covid no Senado'),
 ('PESSOA', 'Fabio Wajngarten'),
 ('ORG', 'Presidência'),
 ('ORG', 'Palácio do Planalto'),
 ('PESSOA', 'Wajngarten'),
 ('ORG', 'BioNTech'),
 ('ORG', 'CPI'),
 ('PESSOA', 'Jair Bolsonaro'),
 ('PESSOA', 'Albert Bourla'),
 ('ORG', 'Pfizer'),
 ('PESSOA', 'Bolsonaro')]

In [20]:
path2save

'/home/cz/mds3/REN/few-shot/OUTPUTS/5shot/Gemini'

In [104]:
np.save(os.path.join(path2save, f'predicted_entities-{3}.npy'), list(d.item()))

In [169]:
path = '/home/cz/mds3/REN/few-shot/OUTPUTS/Zeroshot/Coral/Tips'

In [190]:
for i in range(64):
    print(i)
    with open(os.path.join(path,f'predicted_entities-{i}.txt'), 'r') as archivo:
        # Lee el contenido del archivo
        contenido = archivo.read()
    new_data = caso22(contenido.replace('Aqui está a lista de tuplas com as entidades nomeadas e seus respectivos labels:\n\n',''))
    np.save(os.path.join(path, f'predicted_entities-{i}.npy'),new_data)
    print(new_data)
    if len(new_data)==0:
        print('ZERADO', i)
    

0
[('Arezzo', 'ORG'), ('Soma', 'ORG'), ('Hering', 'ORG'), ('Thiago Hering', 'PESSOA'), ('Thiago', 'PESSOA'), ('Rafael Bossolani', 'PESSOA')]
1
[('ABPA', 'ORG'), ('Rio Grande do Sul', 'LOC'), ('Santa Catarina', 'LOC')]
2
[('Brasil', 'LOC'), ('Washington', 'LOC'), ('EUA', 'LOC'), ('Chile', 'LOC'), ('Argentina', 'LOC'), ('Otaviano Canuto', 'PESSOA'), ('Joe Biden', 'PESSOA'), ('Canuto', 'PESSOA'), ('Banco Mundial', 'ORG'), ('Fundo Monetário Internacional', 'ORG'), ('FMI', 'ORG'), ('Center for Macroeconomics and Development', 'ORG'), ('Ministério da Fazenda', 'ORG'), ('Federal Reserve', 'ORG'), ('Fed', 'ORG'), ('Banco Central', 'ORG'), ('BC', 'ORG'), ('Copom', 'ORG'), ('Tesouro', 'ORG')]
3
[('David Lynch', 'PESSOA'), ('Vânia Mignone', 'PESSOA'), ('Gabriel Pérez-Barreiro', 'PESSOA'), ('Temer', 'PESSOA'), ('Bolsonaro', 'PESSOA'), ('Jacques Lacan', 'PESSOA'), ('Francisco Mignone', 'PESSOA'), ('Nando Reis', 'PESSOA'), ('George Orwell', 'PESSOA'), ('Miguel de Cervantes', 'PESSOA'), ('Campinas', 

In [118]:
for i in range(64):
    print(i)
    with open(os.path.join(path,f'predicted_entities-{i}.txt'), 'r') as archivo:
        # Lee el contenido del archivo
        contenido = archivo.read()
    if i <=50:
        new_data = np.concatenate(caso18_v2(contenido))
        np.save(os.path.join(path, f'predicted_entities-{i}.npy'),new_data)
    else:
        if i==54:
            continue
        elif i==55 or i==57:
            new_data = np.concatenate(caso18_v2(contenido))
            np.save(os.path.join(path, f'predicted_entities-{i}.npy'),new_data)
        else:
            input_string = contenido
            new_data = recab_string_to_tuples(input_string)
            np.save(os.path.join(path, f'predicted_entities-{i}.npy'),new_data)
    print(new_data)
        
        

0
[['Arezzo' 'ORG']
 ['Soma' 'ORG']
 ['Hering' 'ORG']
 ['Hering' 'ORG']
 ['Hering' 'ORG']
 ['Hering' 'ORG']
 ['Soma' 'ORG']
 ['Farm' 'ORG']
 ['Animale' 'ORG']
 ['Hering' 'ORG']
 ['Hering' 'ORG']
 ['Hering' 'ORG']
 ['Thiago Hering' 'PESSOA']
 ['Thiago Hering' 'PESSOA']
 ['Thiago' 'PESSOA']
 ['Rafael Bossolani' 'PESSOA']]
1
[['Yara' 'ORG']
 ['Associação Brasileira de Proteína Animal' 'ORG']
 ['ABPA' 'ORG']
 ['Rio Grande do Sul' 'LOC']
 ['Santa Catarina' 'LOC']]
2
[['Brasil' 'LOC']
 ['Washington' 'LOC']
 ['EUA' 'LOC']
 ['Chile' 'LOC']
 ['Argentina' 'LOC']
 ['Amazônia' 'LOC']
 ['Otaviano Canuto' 'PESSOA']
 ['Canuto' 'PESSOA']
 ['Joe Biden' 'PESSOA']
 ['Banco Mundial' 'ORG']
 ['Fundo Monetário Internacional' 'ORG']
 ['FMI' 'ORG']
 ['Valor' 'ORG']
 ['Center for Macroeconomics and Development' 'ORG']
 ['Ministério da Fazenda' 'ORG']
 ['Fed' 'ORG']
 ['Federal Reserve' 'ORG']
 ['Banco Central' 'ORG']
 ['BC' 'ORG']
 ['Copom' 'ORG']
 ['Tesouro' 'ORG']]
3
[['David Lynch' 'PESSOA']
 ['David' 'PESSO

In [207]:
with open(os.path.join(path,f'predicted_entities-{45}.txt'), 'r') as archivo:
        # Lee el contenido del archivo
        contenido = archivo.read()

In [212]:
contenido.replace('Aqui está uma lista de tuplas contendo as entidades nomeadas encontradas no texto, juntamente com seus respectivos rótulos:\n\n','').replace('-','').replace('\n','')

' (Sabino da Silva Marques, PESSOA) (Marco Aurélio, PESSOA) (Paulo Guedes, PESSOA) (Ney Bello, PESSOA) (Gilmar Mendes, PESSOA) (Fabrício Queiroz, PESSOA) (Márcia Aguiar, PESSOA) (Dario Messer, PESSOA) (Collor, PESSOA) (PC Farias, PESSOA) (Roberto Irineu, PESSOA) (João Roberto Marinho, PESSOA) (Marinho, PESSOA) (Jair Bolsonaro, PESSOA) (Marcelo Daher, PESSOA) (Antonio Palocci, PESSOA) (Lula, PESSOA) (Sergio Moro, PESSOA) (Migalhas, PESSOA) (André Esteves, PESSOA) (Alex Silveira, PESSOA) (Alexandre de Moraes, PESSOA) (Humberto Martins, PESSOA) (Antônio Roberto Andolfatto de Souza, PESSOA) (Marco Antonio Massaneiro, PESSOA) (Leonardo Cocentino, PESSOA) (Silvio Latache, PESSOA) (José Marcelo Fernandes, PESSOA) (Marcelo Crestani Rubel, PESSESSOA) (Gilberto da Graça Couto Filho, PESSOA) (Willer Tomaz, PESSOA) (Renata Oliveira, PESSOA) (Luciana Celidonio, PESSOA) (Fernanda Neves Piva, PESSOA) (Thaís G. Pascoaloto Venturi, PESSOA) (Saul Tourinho Leal, PESSOA) (Bruno Casagrande e Silva, PESSOA)

In [213]:
caso24(contenido.replace('Aqui está uma lista de tuplas contendo as entidades nomeadas encontradas no texto, juntamente com seus respectivos rótulos:\n\n','').replace('-','').replace('\n',''))

[('Sabino da Silva Marques', 'PESSOA'),
 ('Marco Aurélio', 'PESSOA'),
 ('Paulo Guedes', 'PESSOA'),
 ('Ney Bello', 'PESSOA'),
 ('Gilmar Mendes', 'PESSOA'),
 ('Fabrício Queiroz', 'PESSOA'),
 ('Márcia Aguiar', 'PESSOA'),
 ('Dario Messer', 'PESSOA'),
 ('Collor', 'PESSOA'),
 ('PC Farias', 'PESSOA'),
 ('Roberto Irineu', 'PESSOA'),
 ('João Roberto Marinho', 'PESSOA'),
 ('Marinho', 'PESSOA'),
 ('Jair Bolsonaro', 'PESSOA'),
 ('Marcelo Daher', 'PESSOA'),
 ('Antonio Palocci', 'PESSOA'),
 ('Lula', 'PESSOA'),
 ('Sergio Moro', 'PESSOA'),
 ('Migalhas', 'PESSOA'),
 ('André Esteves', 'PESSOA'),
 ('Alex Silveira', 'PESSOA'),
 ('Alexandre de Moraes', 'PESSOA'),
 ('Humberto Martins', 'PESSOA'),
 ('Antônio Roberto Andolfatto de Souza', 'PESSOA'),
 ('Marco Antonio Massaneiro', 'PESSOA'),
 ('Leonardo Cocentino', 'PESSOA'),
 ('Silvio Latache', 'PESSOA'),
 ('José Marcelo Fernandes', 'PESSOA'),
 ('Marcelo Crestani Rubel', 'PESSESSOA'),
 ('Gilberto da Graça Couto Filho', 'PESSOA'),
 ('Willer Tomaz', 'PESSOA'),
 

In [214]:
np.save(os.path.join(path, f'predicted_entities-{45}.npy'),caso24(contenido.replace('Aqui está uma lista de tuplas contendo as entidades nomeadas encontradas no texto, juntamente com seus respectivos rótulos:\n\n','').replace('-','').replace('\n','')))

In [46]:
root = root = '/home/cz/'

In [47]:
import sys
sys.path.append(root+'/mds3/REN/few-shot')
from help2process_coral import*

In [48]:
caso18_v2(contenido)

[[('Arezzo', 'ORG')],
 [('Soma', 'ORG')],
 [('Hering', 'ORG')],
 [('Hering', 'ORG')],
 [('Hering', 'ORG')],
 [('Hering', 'ORG')],
 [('Soma', 'ORG')],
 [('Farm', 'ORG')],
 [('Animale', 'ORG')],
 [('Hering', 'ORG')],
 [('Hering', 'ORG')],
 [('Hering', 'ORG')],
 [('Thiago Hering', 'PESSOA')],
 [('Thiago Hering', 'PESSOA')],
 [('Thiago', 'PESSOA')],
 [('Rafael Bossolani', 'PESSOA')]]

In [115]:
import json

def recab_string_to_tuples(input_string):
    # Carregar a string como um objeto Python (lista de dicionários)
    entidades = json.loads(input_string)
    #print(entidades,'en')
    # Criar uma lista de tuplas (entidade, label)
    try:
        resultado = [(entidade['entidade'].strip(), entidade['label']) for entidade in entidades]
    except:
        resultado = [(entidade['entity'].strip(), entidade['label']) for entidade in entidades]
        
    
    return resultado



# Exemplo de uso
input_string = contenido
resultado = recab_string_to_tuples(input_string)
print(resultado)


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 4 (char 3)

In [162]:
def recab_string_to_tuples(input_string):
    # Remover os colchetes externos e dividir a string
    input_string = input_string.strip('[]')
    
    # Dividir em partes e extrair entidades e rótulos
    entities = []
    for item in input_string.split('},'):
        item = item.strip(' {')
        if '}' in item:
            item = item.rstrip('}')
        # Separar a entidade e o label
        parts = item.split(',')
        if len(parts) == 2:
            entity = parts[0].strip().strip('"')
            label = parts[1].strip().strip('"')
            entities.append((entity, label))

    return entities

# Exemplo de uso
input_string = conteido 
resultado = recab_string_to_tuples(input_string)
print(resultado)


[('[{ "Sara Giromini', 'PESSOA'), ('Sara Winter', 'PESSOA'), ('Sara Winter', 'PESSOA'), ('Sara', 'PESSOA'), ('\\u00c1urea Carolina', 'PESSOA'), ('C\\u00e2mara', 'ORG'), ('Minist\\u00e9rio P\\u00fablico do Distrito Federal', 'ORG'), ('Procuradoria Federal dos Direitos do Cidad\\u00e3o', 'ORG'), ('Recife', 'LOC'), ('PE', 'LOC"}]')]
